In [1]:
import pandas as pd
import json
import numpy as np

Load Base Data

In [2]:
df = pd.read_csv('train.csv')

C:\Users\pecke\AppData\Local\Temp\ipykernel_31076\2436019669.py:1: DtypeWarning: Columns (0: postal) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('train.csv')


In [3]:
df

,id,Tranc_YearMonth,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,...,vacancy,pri_sch_affiliation,pri_sch_latitude,pri_sch_longitude,sec_sch_nearest_dist,sec_sch_name,cutoff_point,affiliation,sec_sch_latitude,sec_sch_longitude
0,88471,2016-05,KALLANG/WHAMPOA,4 ROOM,3B,UPP BOON KENG RD,10 TO 12,90.0,Model A,2006,...,78,1,1.317659,103.882504,1138.633422,Geylang Methodist School,224,0,1.317659,103.882504
1,122598,2012-07,BISHAN,5 ROOM,153,BISHAN ST 13,07 TO 09,130.0,Improved,1987,...,45,1,1.349783,103.854529,447.894399,Kuo Chuan Presbyterian Secondary School,232,0,1.350110,103.854892
2,170897,2013-07,BUKIT BATOK,EXECUTIVE,289B,BT BATOK ST 25,13 TO 15,144.0,Apartment,1997,...,39,0,1.345245,103.756265,180.074558,Yusof Ishak Secondary School,188,0,1.342334,103.760013
3,86070,2012-04,BISHAN,4 ROOM,232,BISHAN ST 22,01 TO 05,103.0,Model A,1992,...,20,1,1.354789,103.844934,389.515528,Catholic High School,253,1,1.354789,103.844934
4,153632,2017-12,YISHUN,4 ROOM,876,YISHUN ST 81,01 TO 03,83.0,Simplified,1987,...,74,0,1.416280,103.838798,312.025435,Orchid Park Secondary School,208,0,1.414888,103.838335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150629,39814,2020-09,WOODLANDS,EXECUTIVE,849,WOODLANDS ST 82,04 TO 06,161.0,Apartment,1995,...,43,0,1.444148,103.794545,189.889876,Evergreen Secondary School,224,0,1.441221,103.793772
150630,147177,2017-06,JURONG WEST,5 ROOM,648D,JURONG WEST ST 61,04 TO 06,110.0,Improved,2001,...,45,0,1.339244,103.698896,614.418470,Boon Lay Secondary School,188,0,1.343224,103.701703
150631,179087,2020-12,BEDOK,EXECUTIVE,639,BEDOK RESERVOIR RD,10 TO 12,144.0,Apartment,1993,...,43,1,1.328471,103.901299,556.889910,Manjusri Secondary School,188,0,1.327520,103.901811
150632,21309,2016-05,QUEENSTOWN,3 ROOM,32,HOLLAND CL,07 TO 09,59.0,Improved,1974,...,82,0,1.299811,103.799965,832.386515,Queensway Secondary School,214,0,1.300475,103.801724


Define Dictionary of the Final Results of the Data Quality. After each test, there will be a key value pair to record the result of that respective test. If the data quality is acceptable, a True will be assigned to the key. If there is a data quality issue, a False will be assigned.

In [4]:
data_cleaning_dict = {}
# True means no action needed
# False means cleaning is needed

Check for Unique IDs

This function checks the the number of rows matches the number of entries

In [5]:
def unique_id():
    if df['id'].count() == df['id'].nunique():
        return True
    else:
        return False

In [6]:
data_cleaning_dict['unique_id'] = unique_id()
data_cleaning_dict

{'unique_id': True}

Check for NULLS in Planning Areas Columns

The following function confirms if the number of nulls in the planning area column is 0.

In [7]:
def planning_area_nulls():
    if df['planning_area'].isna().sum() == 0:
        return True
    else:
        return False

In [8]:
data_cleaning_dict['planning_area_nulls'] = planning_area_nulls()
data_cleaning_dict

{'unique_id': True, 'planning_area_nulls': True}

Check for Consistency between Tranc_YearMonth, Tranc_Year and Tranc_Month

The following code block splits the Year Month Transaction Column into a Year Column and a Month Column as casts them to integers

In [9]:
df_year_month = df
df_year_month[['Tranc_YearMonth_Y', 'Tranc_YearMonth_M']] = df['Tranc_YearMonth'].str.split("-", expand=True)
# Tranc = Tranc.drop(columns=['0', '1'])
df_year_month[['Tranc_YearMonth_Y', 'Tranc_YearMonth_M']] = df[['Tranc_YearMonth_Y', 'Tranc_YearMonth_M']].astype(int)

In [10]:
df_year_month.info()

<class 'pandas.DataFrame'>
RangeIndex: 150634 entries, 0 to 150633
Data columns (total 80 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   id                         150634 non-null  int64  
 1   Tranc_YearMonth            150634 non-null  str    
 2   town                       150634 non-null  str    
 3   flat_type                  150634 non-null  str    
 4   block                      150634 non-null  str    
 5   street_name                150634 non-null  str    
 6   storey_range               150634 non-null  str    
 7   floor_area_sqm             150634 non-null  float64
 8   flat_model                 150634 non-null  str    
 9   lease_commence_date        150634 non-null  int64  
 10  resale_price               150634 non-null  float64
 11  Tranc_Year                 150634 non-null  int64  
 12  Tranc_Month                150634 non-null  int64  
 13  mid_storey                 150634 non-nu

The following function checks if the original column Tranc_Year is the same as the newly made Year Column.
If there are any entries that do not agree with each other, the sum would be greater than 0 (False is the same as 0 and True is the same as 1)

In [11]:
def check_year():
    if (df_year_month['Tranc_Year'] != df_year_month['Tranc_YearMonth_Y']).sum() == 0:
        return True
    else:
        return False

check_year()
# df[df['Tranc_Year'] != df['Tranc_YearMonth_Y']]

True

In [12]:
data_cleaning_dict['check_year'] = check_year()
data_cleaning_dict

{'unique_id': True, 'planning_area_nulls': True, 'check_year': True}

This performs the same function but for the transaction month column.

In [13]:
def check_month():
    if (df_year_month['Tranc_Month'] != df_year_month['Tranc_YearMonth_M']).sum() == 0:
        return True
    else:
        return False

check_month()
# df[df['Tranc_Year'] != df['Tranc_YearMonth_Y']]

True

The conclusion is that there is no inconsistency between the Transaction YearMonth column and the other two columns (Transaction Year and Transaction Month)

In [14]:
data_cleaning_dict['check_month'] = check_month()
data_cleaning_dict

{'unique_id': True,
 'planning_area_nulls': True,
 'check_year': True,
 'check_month': True}

Check Consistency of Lease and Completion Dates

Create a dataframe containing only the lease_commence_date, hdb_age and the year_completed (the construction completion year)

In [15]:
df_start_dates = df[['lease_commence_date', 'hdb_age', 'year_completed']]
df_start_dates

,lease_commence_date,hdb_age,year_completed
0,2006,15,2005
1,1987,34,1987
2,1997,24,1996
3,1992,29,1990
4,1987,34,1987
...,...,...,...
150629,1995,26,1985
150630,2001,20,1998
150631,1993,28,1992
150632,1974,47,1973


The new column checks if the sum of the lease_commence_date and hdb_age adds up to 2021 which is the year of the data set.

In [16]:
df_start_dates['lease_hdb_age'] = (df_start_dates['lease_commence_date'] + df_start_dates['hdb_age'])
df_start_dates

,lease_commence_date,hdb_age,year_completed,lease_hdb_age
0,2006,15,2005,2021
1,1987,34,1987,2021
2,1997,24,1996,2021
3,1992,29,1990,2021
4,1987,34,1987,2021
...,...,...,...,...
150629,1995,26,1985,2021
150630,2001,20,1998,2021
150631,1993,28,1992,2021
150632,1974,47,1973,2021


If there are rows which do not add up to 2021, then those rows should not be trusted, either the lease commence date or the hdb_age could have been wrong.

In [17]:
def lease_hdb_age():
    if (df_start_dates['lease_hdb_age'] != 2021).sum() == 0:
        return True
    else:
        return False


# df_start_dates['lease_hdb_age'] == 2021


In [18]:
data_cleaning_dict['lease_hdb_age'] = lease_hdb_age()
data_cleaning_dict

{'unique_id': True,
 'planning_area_nulls': True,
 'check_year': True,
 'check_month': True,
 'lease_hdb_age': True}

This checks if there are rows where the lease commencement date is earlier than the year of the completion of the construction. Since the function returned False, there are flats that have an inconsistency between the lease commencement date and the year of completion of the construction.

In [ ]:
def compld_lease_commence():
    if (df_start_dates['lease_commence_date'] < df_start_dates['year_completed']).sum() == 0:
        return True
    else:
        return False
# (df_start_dates['lease_commence_date'] < df_start_dates['year_completed']).sum()

np.int64(485)

Since the lease_hdb_age check gave 'True' but construct_lease is 'False', this means that lease_commence_date of each row can be trusted but there are some rows where the year of construction completion are incorrect.

In [22]:
data_cleaning_dict['construct_lease'] = compld_lease_commence()
data_cleaning_dict
# Since lease_hdb_age is True but construct_lease is False, this means that lease_commence_date can be trusted but not all the rows of the construction completion date

{'unique_id': True,
 'planning_area_nulls': True,
 'check_year': True,
 'check_month': True,
 'lease_hdb_age': True,
 'construct_lease': False}

These are the rows that have problematic construction completion dates. It was then decided that the hdb_age column should be used in the model instead. However, during the presentation, it was pointed out that the prices were tagged to the year of transaction, the analysis should have used the age of the flat at the time of transaction instead of the age of the flat in 2021.

In [ ]:

df[df_start_dates['lease_commence_date'] < df_start_dates['year_completed']]

,id,Tranc_YearMonth,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,...,pri_sch_latitude,pri_sch_longitude,sec_sch_nearest_dist,sec_sch_name,cutoff_point,affiliation,sec_sch_latitude,sec_sch_longitude,Tranc_YearMonth_Y,Tranc_YearMonth_M
469,66467,2020-11,CLEMENTI,3 ROOM,311A,CLEMENTI AVE 4,04 TO 06,60.0,DBSS,2014,...,1.316148,103.767576,554.939903,Clementi Town Secondary School,231,0,1.315475,103.762079,2020,11
712,48004,2017-01,JURONG EAST,3 ROOM,37,TEBAN GDNS RD,04 TO 06,67.0,Improved,1966,...,1.312622,103.757030,378.630976,Commonwealth Secondary School,237,0,1.319128,103.745756,2017,1
959,65501,2020-07,BEDOK,3 ROOM,203,BEDOK NTH ST 1,04 TO 06,68.0,New Generation,1977,...,1.330325,103.931885,942.077630,Ping Yi Secondary School,189,0,1.327140,103.920836,2020,7
1239,162459,2014-09,TAMPINES,5 ROOM,515B,TAMPINES CTRL 7,07 TO 09,108.0,DBSS,2008,...,1.358087,103.935352,709.954241,Junyuan Secondary School,188,0,1.353341,103.933411,2014,9
1627,139788,2021-03,BUKIT MERAH,4 ROOM,57,TELOK BLANGAH HTS,01 TO 03,91.0,New Generation,1976,...,1.276130,103.808639,1095.743326,CHIJ Saint Theresa's Convent,235,0,1.276029,103.822344,2021,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149013,164801,2014-05,TAMPINES,5 ROOM,515C,TAMPINES CTRL 7,01 TO 03,107.0,DBSS,2008,...,1.358087,103.935352,739.394429,Junyuan Secondary School,188,0,1.353341,103.933411,2014,5
149812,66498,2020-07,CLEMENTI,5 ROOM,311A,CLEMENTI AVE 4,04 TO 06,105.0,DBSS,2014,...,1.316148,103.767576,554.939903,Clementi Town Secondary School,231,0,1.315475,103.762079,2020,7
150256,48009,2020-01,JURONG EAST,3 ROOM,37,TEBAN GDNS RD,04 TO 06,67.0,Improved,1966,...,1.312622,103.757030,378.630976,Commonwealth Secondary School,237,0,1.319128,103.745756,2020,1
150287,164782,2016-06,TAMPINES,5 ROOM,515C,TAMPINES CTRL 7,10 TO 12,108.0,DBSS,2008,...,1.358087,103.935352,739.394429,Junyuan Secondary School,188,0,1.353341,103.933411,2016,6


Storeys Consistency

This code block splits the 'storey_range' column into the respective upper and lower ranges and casts them as integers

In [22]:
df_storey = df
df_storey[['storey_lower', 'storey_upper']] = df['storey_range'].str.split(" TO ", expand=True)
# Tranc = Tranc.drop(columns=['0', '1'])

df_storey[['storey_lower', 'storey_upper']] = df_storey[['storey_lower', 'storey_upper']].astype(int)
df_storey = df_storey[['storey_lower', 'lower', 'storey_upper', 'upper', 'mid']]
df_storey

,storey_lower,lower,storey_upper,upper,mid
0,10,10,12,12,11
1,7,7,9,9,8
2,13,13,15,15,14
3,1,1,5,5,3
4,1,1,3,3,2
...,...,...,...,...,...
150629,4,4,6,6,5
150630,4,4,6,6,5
150631,10,10,12,12,11
150632,7,7,9,9,8


The following function checks if there is consistency between the existing 'lower' column which indicates the lower range of the floor level of that flat and the information given in the 'storey_range' column. The dataset gives this duplication of information, if this information does not tally then the information cannot be trusted.

In [ ]:
def storey_lower_check():
    if (df_storey['storey_lower'] != df_storey['lower']).sum() == 0:
        return True
    else:
        return False


(df_storey['storey_lower'] != df_storey['lower']).count()
storey_lower_check()

True

In [116]:
data_cleaning_dict['storey_lower'] = storey_lower_check()
data_cleaning_dict

{'unique_id': True,
 'planning_area_nulls': True,
 'check_year': True,
 'check_month': True,
 'lease_hdb_age': True,
 'construct_lease': False,
 'bus_interchange': True,
 'storey_lower': True}

The following function performs the same check as the function above but for the upper range given for each flat.

In [ ]:
def storey_upper_check():
    if (df_storey['storey_upper'] != df_storey['upper']).sum() == 0:
        return True
    else:
        return False


True

In [122]:
data_cleaning_dict['storey_upper'] = storey_upper_check()
data_cleaning_dict

{'unique_id': True,
 'planning_area_nulls': True,
 'check_year': True,
 'check_month': True,
 'lease_hdb_age': True,
 'construct_lease': False,
 'storey_lower': True,
 'storey_upper': True}

This code block checks the column 'mid' which indicates the middle of the upper and lower range of the flat. This 'mid' column is compared with the calculated average of the 'upper' and 'lower' columns to see if there is consistency between all three columns.

In [29]:
df_storey['average'] = (df_storey['upper'] + df_storey['lower']) / 2
# df_storey['mid']
df_storey['average'] = df_storey['average'].astype(int)
df_storey

,storey_lower,lower,storey_upper,upper,mid,average
0,10,10,12,12,11,11
1,7,7,9,9,8,8
2,13,13,15,15,14,14
3,1,1,5,5,3,3
4,1,1,3,3,2,2
...,...,...,...,...,...,...
150629,4,4,6,6,5,5
150630,4,4,6,6,5,5
150631,10,10,12,12,11,11
150632,7,7,9,9,8,8


In [31]:
def storey_mid_check():
    if df_storey['mid'].sum() == df_storey['average'].sum():
        return True
    else:
        return False

storey_mid_check()

True

Final Results of the Data Quality. Since there is a data quality issue of the year of construction, the hdb age was used instead

In [123]:
data_cleaning_dict

{'unique_id': True,
 'planning_area_nulls': True,
 'check_year': True,
 'check_month': True,
 'lease_hdb_age': True,
 'construct_lease': False,
 'storey_lower': True,
 'storey_upper': True}

In [27]:
df[(df['town'] == 'KALLANG/WHAMPOA') & (df['planning_area'] == 'Bukit Batok')]

,id,Tranc_YearMonth,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,...,pri_sch_latitude,pri_sch_longitude,sec_sch_nearest_dist,sec_sch_name,cutoff_point,affiliation,sec_sch_latitude,sec_sch_longitude,Tranc_YearMonth_Y,Tranc_YearMonth_M
2046,136182,2013-01,KALLANG/WHAMPOA,5 ROOM,10,JLN BATU,10 TO 12,127.0,Improved,1986,...,1.366758,103.767695,1010.635595,Assumption English School,188,0,1.368534,103.766675,2013,1
3656,136184,2014-04,KALLANG/WHAMPOA,5 ROOM,10,JLN BATU,07 TO 09,129.0,Improved,1986,...,1.366758,103.767695,1010.635595,Assumption English School,188,0,1.368534,103.766675,2014,4
5961,12899,2012-04,KALLANG/WHAMPOA,3 ROOM,4,JLN BATU,06 TO 10,60.0,Standard,1969,...,1.366758,103.767695,990.526119,Assumption English School,188,0,1.368534,103.766675,2012,4
7278,13065,2019-11,KALLANG/WHAMPOA,3 ROOM,6,JLN BATU,10 TO 12,60.0,Standard,1969,...,1.366758,103.767695,997.278013,Assumption English School,188,0,1.368534,103.766675,2019,11
9937,136172,2017-03,KALLANG/WHAMPOA,5 ROOM,10,JLN BATU,04 TO 06,131.0,Improved,1986,...,1.366758,103.767695,1010.635595,Assumption English School,188,0,1.368534,103.766675,2017,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138787,136179,2021-01,KALLANG/WHAMPOA,5 ROOM,10,JLN BATU,07 TO 09,132.0,Improved,1986,...,1.366758,103.767695,1010.635595,Assumption English School,188,0,1.368534,103.766675,2021,1
139491,48188,2013-11,KALLANG/WHAMPOA,3 ROOM,2,JLN BATU,07 TO 09,60.0,Standard,1969,...,1.366758,103.767695,983.930044,Assumption English School,188,0,1.368534,103.766675,2013,11
142907,13058,2016-10,KALLANG/WHAMPOA,3 ROOM,6,JLN BATU,07 TO 09,60.0,Standard,1969,...,1.366758,103.767695,997.278013,Assumption English School,188,0,1.368534,103.766675,2016,10
144355,12907,2014-02,KALLANG/WHAMPOA,3 ROOM,4,JLN BATU,04 TO 06,60.0,Standard,1969,...,1.366758,103.767695,990.526119,Assumption English School,188,0,1.368534,103.766675,2014,2


Through the inspection of the data, there are 77 rows whereby the town and the planning area did not match. The flat cannot be in the Kallang/Whampoa area and also be in Bukit Batok. Hence, an 'Included' Column was created to indicate that inconsistent rows should not be included in the final analysis and the in training of the model. This dataframe is then saved as a csv file.

In [150]:
df["Included"] = np.where(((df['town'] == 'KALLANG/WHAMPOA') & (df['planning_area'] == 'Bukit Batok')), 0, 1)
df
df.to_csv('train_cleaned.csv', index=False)

In [ ]:
# df_not = df[(df['town'] == 'KALLANG/WHAMPOA') & (df['planning_area'] == 'Bukit Batok')]
# df_not['Included'].sum()


np.int64(0)

In [146]:
df_cleaned = pd.read_csv('train_cleaned.csv')

C:\Users\pecke\AppData\Local\Temp\ipykernel_40044\785055108.py:1: DtypeWarning: Columns (0: postal) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cleaned = pd.read_csv('train_cleaned.csv')


In [148]:
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 150634 entries, 0 to 150633
Data columns (total 80 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Unnamed: 0                 150634 non-null  int64  
 1   id                         150634 non-null  int64  
 2   Tranc_YearMonth            150634 non-null  str    
 3   town                       150634 non-null  str    
 4   flat_type                  150634 non-null  str    
 5   block                      150634 non-null  str    
 6   street_name                150634 non-null  str    
 7   storey_range               150634 non-null  str    
 8   floor_area_sqm             150634 non-null  float64
 9   flat_model                 150634 non-null  str    
 10  lease_commence_date        150634 non-null  int64  
 11  resale_price               150634 non-null  float64
 12  Tranc_Year                 150634 non-null  int64  
 13  Tranc_Month                150634 non-nu

In [149]:
df_cleaned[df_cleaned['Included'] == 1]

,Unnamed: 0,id,Tranc_YearMonth,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,...,pri_sch_affiliation,pri_sch_latitude,pri_sch_longitude,sec_sch_nearest_dist,sec_sch_name,cutoff_point,affiliation,sec_sch_latitude,sec_sch_longitude,Included
0,0,88471,2016-05,KALLANG/WHAMPOA,4 ROOM,3B,UPP BOON KENG RD,10 TO 12,90.0,Model A,...,1,1.317659,103.882504,1138.633422,Geylang Methodist School,224,0,1.317659,103.882504,1
1,1,122598,2012-07,BISHAN,5 ROOM,153,BISHAN ST 13,07 TO 09,130.0,Improved,...,1,1.349783,103.854529,447.894399,Kuo Chuan Presbyterian Secondary School,232,0,1.350110,103.854892,1
2,2,170897,2013-07,BUKIT BATOK,EXECUTIVE,289B,BT BATOK ST 25,13 TO 15,144.0,Apartment,...,0,1.345245,103.756265,180.074558,Yusof Ishak Secondary School,188,0,1.342334,103.760013,1
3,3,86070,2012-04,BISHAN,4 ROOM,232,BISHAN ST 22,01 TO 05,103.0,Model A,...,1,1.354789,103.844934,389.515528,Catholic High School,253,1,1.354789,103.844934,1
4,4,153632,2017-12,YISHUN,4 ROOM,876,YISHUN ST 81,01 TO 03,83.0,Simplified,...,0,1.416280,103.838798,312.025435,Orchid Park Secondary School,208,0,1.414888,103.838335,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150629,150629,39814,2020-09,WOODLANDS,EXECUTIVE,849,WOODLANDS ST 82,04 TO 06,161.0,Apartment,...,0,1.444148,103.794545,189.889876,Evergreen Secondary School,224,0,1.441221,103.793772,1
150630,150630,147177,2017-06,JURONG WEST,5 ROOM,648D,JURONG WEST ST 61,04 TO 06,110.0,Improved,...,0,1.339244,103.698896,614.418470,Boon Lay Secondary School,188,0,1.343224,103.701703,1
150631,150631,179087,2020-12,BEDOK,EXECUTIVE,639,BEDOK RESERVOIR RD,10 TO 12,144.0,Apartment,...,1,1.328471,103.901299,556.889910,Manjusri Secondary School,188,0,1.327520,103.901811,1
150632,150632,21309,2016-05,QUEENSTOWN,3 ROOM,32,HOLLAND CL,07 TO 09,59.0,Improved,...,0,1.299811,103.799965,832.386515,Queensway Secondary School,214,0,1.300475,103.801724,1
